In [1]:
#pip install pandas numpy scikit-learn matplotlib seaborn yfinance lightgbm xgboost bt streamlit joblib

In [2]:
#pip install nselib pandas

In [3]:
#!pip install nselib pandas_market_calendars pandas requests

In [4]:
import numpy as np
import pandas as pd

print(np.__version__)
print(pd.__version__)


1.24.4
1.5.3


### Phase 1: Data Collection (Nifty 100 - Last 5 years)

In [12]:
# ==========================================
# 📊 Phase 1: Data Collection (NIFTY 100)
# Source: NSE India (via nselib)
# ==========================================

from nselib import capital_market
import pandas as pd
import os
from datetime import date
import time

In [13]:
# --- Sentiment model (loaded once) ---
from transformers import pipeline
import feedparser
from datetime import datetime

sentiment_model = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    framework="pt"   # 🔥 force PyTorch, bypass TensorFlow entirely
)

Device set to use cpu


In [14]:
# --- STEP 1: Fetch NIFTY 100 tickers ---
url = "https://archives.nseindia.com/content/indices/ind_nifty100list.csv"
df_list = pd.read_csv(url)
tickers = df_list["Symbol"].tolist()
print(f"✅ Total tickers fetched: {len(tickers)}")

✅ Total tickers fetched: 101


In [15]:
# --- STEP 2: Create data folder ---
os.makedirs("data/raw", exist_ok=True)

In [16]:
# --- STEP 2B: Create news data folder ---
from urllib.parse import quote_plus

NEWS_DIR = "data/news_raw"
os.makedirs(NEWS_DIR, exist_ok=True)


def fetch_news_sentiment(symbol, max_items=50):
    """
    Fetch news headlines and compute sentiment for a stock.
    """
    query = f"{symbol} stock site:reuters.com OR site:economictimes.com OR site:moneycontrol.com"
    query_encoded = quote_plus(query)

    url = f"https://news.google.com/rss/search?q={query_encoded}&hl=en-IN&gl=IN&ceid=IN:en"
    feed = feedparser.parse(url)

    records = []

    for entry in feed.entries[:max_items]:
        text = entry.title

        # Some entries may not have published date
        if not hasattr(entry, "published_parsed"):
            continue

        date_pub = datetime(*entry.published_parsed[:6])

        try:
            sent = sentiment_model(text)[0]
            score = sent["score"] if sent["label"] == "positive" else -sent["score"]
        except Exception:
            continue

        records.append({
            "ticker": symbol,
            "date": date_pub.date(),
            "headline": text,
            "sentiment_score": score
        })

    return pd.DataFrame(records)


In [17]:
# --- STEP 3: Fetch function ---
def fetch_all_nselib_data(symbols, start=date(2020, 1, 1), end=date(2025, 1, 1)):
    failed = []
    for symbol in symbols:
        try:
            print(f"📦 Fetching {symbol} ...")

            # --- Price data ---
            df = capital_market.price_volume_and_deliverable_position_data(
                symbol=symbol,
                from_date=start.strftime("%d-%m-%Y"),
                to_date=end.strftime("%d-%m-%Y")
            )

            if not df.empty:
                df.to_csv(f"data/raw/{symbol}.csv", index=False)
                print(f"✅ Saved: {symbol}")
            else:
                print(f"⚠️ No price data for {symbol}")
                failed.append(symbol)

            # --- News sentiment ---
            news_df = fetch_news_sentiment(symbol)

            if not news_df.empty:
                news_df.to_csv(f"{NEWS_DIR}/{symbol}_news.csv", index=False)
                print(f"📰 News sentiment saved: {symbol}")

        except Exception as e:
            print(f"❌ Failed: {symbol} | {e}")
            failed.append(symbol)

        time.sleep(1)
    
    # --- Summary ---
    print("\n✅ Completed Data Fetching!")
    print(f"✅ Successful: {len(symbols) - len(failed)} | ❌ Failed: {len(failed)}")
    if failed:
        pd.DataFrame(failed, columns=["Symbol"]).to_csv("data/failed_tickers.csv", index=False)
        print("⚠️ Failed tickers saved to failed_tickers.csv")

In [18]:
# --- STEP 4: Execute ---
fetch_all_nselib_data(tickers)

print("\n🎯 All data saved in 'data/raw/' — ready for Phase 2.")

📦 Fetching ABB ...
✅ Saved: ABB
📰 News sentiment saved: ABB
📦 Fetching ADANIENSOL ...
✅ Saved: ADANIENSOL
📰 News sentiment saved: ADANIENSOL
📦 Fetching ADANIENT ...
✅ Saved: ADANIENT
📰 News sentiment saved: ADANIENT
📦 Fetching ADANIGREEN ...
✅ Saved: ADANIGREEN
📰 News sentiment saved: ADANIGREEN
📦 Fetching ADANIPORTS ...
✅ Saved: ADANIPORTS
📰 News sentiment saved: ADANIPORTS
📦 Fetching ADANIPOWER ...
✅ Saved: ADANIPOWER
📰 News sentiment saved: ADANIPOWER
📦 Fetching AMBUJACEM ...
✅ Saved: AMBUJACEM
📰 News sentiment saved: AMBUJACEM
📦 Fetching APOLLOHOSP ...
✅ Saved: APOLLOHOSP
📰 News sentiment saved: APOLLOHOSP
📦 Fetching ASIANPAINT ...
✅ Saved: ASIANPAINT
📰 News sentiment saved: ASIANPAINT
📦 Fetching DMART ...
✅ Saved: DMART
📰 News sentiment saved: DMART
📦 Fetching AXISBANK ...
✅ Saved: AXISBANK
📰 News sentiment saved: AXISBANK
📦 Fetching BAJAJ-AUTO ...
✅ Saved: BAJAJ-AUTO
📰 News sentiment saved: BAJAJ-AUTO
📦 Fetching BAJFINANCE ...
✅ Saved: BAJFINANCE
📰 News sentiment saved: BAJFINANC

### Phase 2: Feature Engineering

In [27]:
# =====================================================
# 🧠 PHASE 2 — FEATURE ENGINEERING
# =====================================================

import numpy as np
from tqdm import tqdm
import logging

In [28]:
# -----------------------------
# Step 1: Setup paths & logging
# -----------------------------
DATA_DIR = r"C:\Users\jayan\Study Materials\ml_alpha_portfolio\data\raw"
FEATURE_DIR = r"C:\Users\jayan\Study Materials\ml_alpha_portfolio\data\features"

os.makedirs(FEATURE_DIR, exist_ok=True)
logging.basicConfig(level=logging.WARNING)

print(f"📈 Found {len(os.listdir(DATA_DIR))} files in {DATA_DIR}")

📈 Found 99 files in C:\Users\jayan\Study Materials\ml_alpha_portfolio\data\raw


In [29]:
# --------------------------------------------------------
# Step 2: Define helper functions for feature engineering
# --------------------------------------------------------

def build_features(df):
    """Compute key features: momentum, volatility, RSI, MACD, EMA cross."""
    df = df.copy()
    df.sort_values('Date', inplace=True)

    # --- Price returns ---
    df['ret_1d'] = df['Close'].pct_change()
    df['momentum_1m'] = df['Close'] / df['Close'].shift(21) - 1
    df['momentum_3m'] = df['Close'] / df['Close'].shift(63) - 1
    df['momentum_6m'] = df['Close'] / df['Close'].shift(126) - 1

    # --- Volatility ---
    df['vol_30d'] = df['ret_1d'].rolling(30).std()
    df['vol_90d'] = df['ret_1d'].rolling(90).std()

    # --- RSI (14-day) ---
    delta = df['Close'].diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)
    roll_up = pd.Series(gain, index=df.index).rolling(14).mean()
    roll_down = pd.Series(loss, index=df.index).rolling(14).mean()
    rs = roll_up / roll_down
    df['rsi_14'] = 100 - (100 / (1 + rs))

    # --- MACD ---
    ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema_12 - ema_26
    df['signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist'] = df['macd'] - df['signal']

    # --- EMA crossover ---
    ema_short = df['Close'].ewm(span=10, adjust=False).mean()
    ema_long = df['Close'].ewm(span=50, adjust=False).mean()
    df['ema_cross'] = (ema_short > ema_long).astype(int)

    # ❗ DO NOT drop NaNs here
    return df


def normalize_features(df, factor_cols):
    """Z-score normalize the factors per date (cross-sectionally)."""
    df = df.copy()
    for col in factor_cols:
        if col in df.columns:
            df[col + '_z'] = df.groupby('Date')[col].transform(lambda x: (x - x.mean()) / x.std())
    return df

NEWS_DIR = r"C:\Users\jayan\Study Materials\ml_alpha_portfolio\data\news_raw"

def load_and_aggregate_sentiment(symbol):
    """
    Load headline-level sentiment and aggregate to daily features.
    """
    news_file = os.path.join(NEWS_DIR, f"{symbol}_news.csv")
    if not os.path.exists(news_file):
        return None

    df = pd.read_csv(news_file)
    df['date'] = pd.to_datetime(df['date'])

    sent_agg = df.groupby('date').agg(
        sent_mean=('sentiment_score', 'mean'),
        sent_sum=('sentiment_score', 'sum'),
        sent_std=('sentiment_score', 'std'),
        news_count=('sentiment_score', 'count')
    ).reset_index()

    return sent_agg


In [31]:
# ---------------------------------------------------
# Step 3: Read each file, clean, build, and normalize
# ---------------------------------------------------

all_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
print(f"📈 Processing {len(all_files)} tickers from: {DATA_DIR}")

for file in tqdm(all_files):
    try:
        file_path = os.path.join(DATA_DIR, file)
        symbol = file.replace(".csv", "")

        # --- Read CSV ---
        df = pd.read_csv(file_path)

        # --- Clean column names ---
        df.columns = [c.strip() for c in df.columns]

        # --- Rename NSE-style columns ---
        col_map = {
            'Date': 'Date',
            'OpenPrice': 'Open',
            'HighPrice': 'High',
            'LowPrice': 'Low',
            'ClosePrice': 'Close',
            'TotalTradedQuantity': 'Volume'
        }
        df.rename(columns=col_map, inplace=True)

        # --- Ensure required columns exist ---
        required_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
        for c in required_cols:
            if c not in df.columns:
                raise ValueError(f"{file} missing column: {c}")

        # --- Parse Date FIRST (critical for sentiment merge) ---
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df.dropna(subset=['Date'], inplace=True)

        # --- Convert numeric columns ---
        for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(',', '', regex=False)
                .astype(float)
            )

        df.sort_values('Date', inplace=True)

        # --- Load & merge sentiment ---
        sent_df = load_and_aggregate_sentiment(symbol)

        if sent_df is not None:
            df = df.merge(
                sent_df,
                left_on='Date',
                right_on='date',
                how='left'
            ).drop(columns=['date'])
        
            # ✅ SENTIMENT ROBUSTNESS FIX (ADD HERE)
            df[['sent_mean', 'sent_sum', 'sent_std']] = (df[['sent_mean', 'sent_sum', 'sent_std']].fillna(0))
            df['news_count'] = df['news_count'].fillna(0)
        
        else:
            df['sent_mean'] = 0.0
            df['sent_sum'] = 0.0
            df['sent_std'] = 0.0
            df['news_count'] = 0

        # --- Compute price-based features ---
        df_feat = build_features(df)

        # --- Sentiment rolling features ---
        df_feat['sent_3d'] = df_feat['sent_mean'].rolling(3).mean()
        df_feat['sent_5d'] = df_feat['sent_mean'].rolling(5).mean()
        df_feat['sent_lag1'] = df_feat['sent_mean'].shift(1)
        df_feat['news_intensity'] = df_feat['news_count'].rolling(5).sum()

        factor_cols = [
            'momentum_1m', 'momentum_3m', 'momentum_6m',
            'vol_30d', 'vol_90d',
            'rsi_14', 'macd_hist', 'ema_cross',
            'sent_3d', 'sent_5d', 'sent_lag1', 'news_intensity'
        ]

        # --- Cross-sectional normalization ---
        df_feat = normalize_features(df_feat, factor_cols)

        # --- Controlled NaN removal (core alpha features only) ---
        min_required_cols = [
            'momentum_6m',
            'vol_90d',
            'rsi_14',
            'macd_hist'
        ]
        
        df_feat.dropna(subset=min_required_cols, inplace=True)

        
        # --- Save engineered features ---
        out_path = os.path.join(FEATURE_DIR, file)
        df_feat.to_csv(out_path, index=False)

    except Exception as e:
        logging.warning(f"❌ Failed for {file}: {e}")
        continue

print("✅ Feature engineering completed for all tickers!")


📈 Processing 99 tickers from: C:\Users\jayan\Study Materials\ml_alpha_portfolio\data\raw


100%|██████████| 99/99 [07:41<00:00,  4.66s/it]

✅ Feature engineering completed for all tickers!


In [32]:
# -----------------------------
# Step 4: Sanity check summary
# -----------------------------

sample_file = os.path.join(FEATURE_DIR, os.listdir(FEATURE_DIR)[0])
df_sample = pd.read_csv(sample_file)
print("\n🧩 Sample engineered dataset preview:")
print(df_sample.head(10)[['Date', 'Close', 'momentum_1m', 'vol_30d', 'rsi_14', 'macd_hist', 'ema_cross', 'sent_3d', 
                          'news_intensity']])


🧩 Sample engineered dataset preview:
         Date   Close  momentum_1m   vol_30d     rsi_14  macd_hist  ema_cross  \
0  2020-07-06  983.40     0.120684  0.025284  95.072949  16.141388          1   
1  2020-07-07  961.40     0.094428  0.025825  90.033975  13.235116          1   
2  2020-07-08  944.65     0.107704  0.026273  83.486043   9.415125          1   
3  2020-07-09  928.50     0.118203  0.026683  75.632490   5.239514          1   
4  2020-07-10  916.95     0.131967  0.026940  69.455298   1.341520          1   
5  2020-07-13  914.90     0.133143  0.019898  65.237249  -1.574921          1   
6  2020-07-14  908.15     0.126597  0.020002  60.560797  -4.019611          1   
7  2020-07-15  905.80     0.139443  0.019942  57.008310  -5.750503          1   
8  2020-07-16  905.85     0.154464  0.019935  47.102931  -6.776995          1   
9  2020-07-17  912.15     0.158286  0.017786  46.482759  -6.893867          1   

   sent_3d  news_intensity  
0      0.0             0.0  
1      0.0  

In [33]:
sample_file

'C:\\Users\\jayan\\Study Materials\\ml_alpha_portfolio\\data\\features\\ABB.csv'

In [35]:
df_sample.head(10)

,Symbol,Series,Date,PrevClose,Open,High,Low,LastPrice,Close,AveragePrice,...,momentum_6m_z,vol_30d_z,vol_90d_z,rsi_14_z,macd_hist_z,ema_cross_z,sent_3d_z,sent_5d_z,sent_lag1_z,news_intensity_z
0,ABB,EQ,2020-07-06,979.50,982.0,1017.00,976.20,981.25,983.40,995.16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABB,EQ,2020-07-07,983.40,987.5,988.95,957.25,959.00,961.40,964.78,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABB,EQ,2020-07-08,961.40,968.0,971.50,942.00,943.30,944.65,952.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABB,EQ,2020-07-09,944.65,947.0,955.00,926.05,928.50,928.50,935.78,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABB,EQ,2020-07-10,928.50,930.0,936.75,914.90,915.15,916.95,922.61,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ABB,EQ,2020-07-13,916.95,922.0,925.35,910.00,913.00,914.90,915.72,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ABB,EQ,2020-07-14,914.90,913.0,915.00,901.05,910.50,908.15,909.57,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ABB,EQ,2020-07-15,908.15,916.0,933.00,900.90,906.10,905.80,915.57,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ABB,EQ,2020-07-16,905.80,909.9,919.00,894.00,909.95,905.85,906.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ABB,EQ,2020-07-17,905.85,914.0,923.25,907.00,914.50,912.15,914.18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Phase 3: Modelling

In [61]:
# =====================================================
# 🧠 PHASE 3 — MODELLING
# =====================================================
# Can run after a kernel restart
# Loads features, benchmark, cleans, computes targets
# Runs rolling LightGBM regression CV

In [16]:
import os
import numpy as np
import pandas as pd
import glob
import requests
import time
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import roc_auc_score

In [17]:
# ==========================================================
# ---------------------- CONFIG -----------------------------
# ==========================================================

BASE_DIR = r"C:\Users\jayan\Study Materials\ml_alpha_portfolio"
DATA_DIR = os.path.join(BASE_DIR, "data")
FEATURE_DIR = os.path.join(DATA_DIR, "features")
MODELS_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0"}

In [18]:
# ==========================================================
# 🔹 STEP 1: LOAD PRICE FEATURE PANEL
# ==========================================================

print("📥 STEP 1: Loading feature CSVs...")

dfs = []
for fp in glob.glob(os.path.join(FEATURE_DIR, "*.csv")):
    df = pd.read_csv(fp)
    df.columns = df.columns.str.strip().str.lower()
    if 'ticker' not in df.columns:
        df['ticker'] = os.path.basename(fp).split(".")[0]
    dfs.append(df)

panel = pd.concat(dfs, ignore_index=True)
panel['date'] = pd.to_datetime(panel['date'], errors='coerce')
panel = panel.dropna(subset=['date', 'ticker'])
panel = panel.sort_values(['ticker', 'date'])

print(f"✅ Panel loaded: {len(panel)} rows")

📥 STEP 1: Loading feature CSVs...
✅ Panel loaded: 150940 rows


In [19]:
# ==========================================================
# 🔹 STEP 2: CREATE TARGET
# ==========================================================

print("🎯 STEP 2: Creating forward return target...")

panel['fwd_ret'] = (
    panel.groupby('ticker')['close'].shift(-5) / panel['close'] - 1
)

panel['target'] = (panel['fwd_ret'] > 0).astype(int)
panel = panel.dropna(subset=['target'])

print("✅ Target ready")


🎯 STEP 2: Creating forward return target...
✅ Target ready


In [20]:
# ==========================================================
# 🔹 STEP 3: EXTRACT CONCALL HIGHLIGHTS
# ==========================================================

print("🗣 STEP 3: Scraping concall highlights...")

def scrape_concall(ticker):
    url = f"https://www.screener.in/company/{ticker}/concall/"
    r = requests.get(url, headers=HEADERS)
    if r.status_code != 200:
        return None
    
    soup = BeautifulSoup(r.text, "html.parser")
    blocks = soup.find_all("div", class_="concall-highlight")
    
    records = []
    for b in blocks:
        try:
            date = pd.to_datetime(b.find("span").text.strip(), errors="coerce")
            text = b.get_text().strip()
            records.append({"ticker": ticker, "date": date, "text": text})
        except:
            continue
    
    return pd.DataFrame(records)

tickers = panel['ticker'].unique()
concall_list = []

for t in tickers:
    df = scrape_concall(t)
    if df is not None:
        concall_list.append(df)
    time.sleep(1)

if len(concall_list) > 0:
    concall_df = pd.concat(concall_list, ignore_index=True)
else:
    concall_df = pd.DataFrame(columns=["ticker","date","text"])

print("✅ Concall scraping complete")


🗣 STEP 3: Scraping concall highlights...
✅ Concall scraping complete


In [21]:
# ==========================================================
# 🔹 STEP 4: NLP FEATURE ENGINEERING
# ==========================================================

print("🧠 STEP 4: Creating guidance delta & confidence index...")

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

concall_df['clean_text'] = concall_df['text'].apply(clean_text)

positive_words = ["strong","growth","robust","improved","confident"]
negative_words = ["weak","pressure","decline","uncertain","slowdown"]

def sentiment_score(text):
    pos = sum(word in text for word in positive_words)
    neg = sum(word in text for word in negative_words)
    return pos - neg

concall_df['raw_sentiment'] = concall_df['clean_text'].apply(sentiment_score)

concall_df = concall_df.sort_values(['ticker','date'])

concall_df['guidance_delta'] = (
    concall_df.groupby('ticker')['raw_sentiment'].diff()
).fillna(0)

confidence_words = ["confident","visibility","strong"]
hedge_words = ["may","might","uncertain","challenge"]

def confidence_score(text):
    pos = sum(word in text for word in confidence_words)
    neg = sum(word in text for word in hedge_words)
    return pos - neg

concall_df['confidence_index'] = concall_df['clean_text'].apply(confidence_score)


🧠 STEP 4: Creating guidance delta & confidence index...


In [22]:
# ==========================================================
# 🔹 STEP 5: MERGE CONCALL FEATURES INTO PANEL
# ==========================================================

print("🔗 STEP 5: Merging concall features...")

concall_features = concall_df[
    ['ticker','date','guidance_delta','confidence_index']
]

panel = panel.merge(concall_features, on=['ticker','date'], how='left')

panel[['guidance_delta','confidence_index']] = (
    panel.groupby('ticker')[['guidance_delta','confidence_index']].ffill()
)

panel[['guidance_delta','confidence_index']] = (
    panel[['guidance_delta','confidence_index']].fillna(0)
)

print("✅ Concall features merged")

🔗 STEP 5: Merging concall features...
✅ Concall features merged


In [23]:
# ==========================================================
# 🔹 STEP 6: FEATURE BLOCKS
# ==========================================================

print("📊 STEP 6: Building feature blocks...")

technical_features = [
    c for c in panel.columns
    if any(x in c for x in ['rsi','macd','momentum','roc','stoch'])
]

price_structure_features = [
    c for c in panel.columns
    if any(x in c for x in ['volatility','atr','range','volume','beta'])
]

concall_features = ['guidance_delta']
confidence_features = ['confidence_index']

hybrid_features = list(set(
    technical_features +
    price_structure_features +
    concall_features +
    confidence_features
))

print("Technical:", len(technical_features))
print("Price Structure:", len(price_structure_features))
print("Concall:", len(concall_features))
print("Confidence:", len(confidence_features))
print("Total Hybrid:", len(hybrid_features))

📊 STEP 6: Building feature blocks...
Technical: 11
Price Structure: 1
Concall: 1
Confidence: 1
Total Hybrid: 14


In [26]:
# ==========================================================
# 🔹 STEP 7: ROLLING CROSS VALIDATION
# ==========================================================

print("🔄 STEP 7: Running rolling CV...")

import warnings
import lightgbm as lgb
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

LGB_PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,        # 🔥 suppress LightGBM warnings
    force_col_wise=True  # 🔥 remove row/col threading message
)

def rolling_train(panel, features, min_train_days=120, test_window=20):

    results = []
    dates = sorted(panel['date'].unique())

    for i in tqdm(range(min_train_days, len(dates) - test_window)):

        train = panel[panel['date'] <= dates[i]]
        test = panel[
            (panel['date'] > dates[i]) &
            (panel['date'] <= dates[i + test_window])
        ]

        if train.empty or test.empty:
            continue

        X_train = train[features]
        y_train = train['target']

        X_test = test[features]
        y_test = test['target']

        # Median imputation
        medians = X_train.median()
        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        model = LGBMClassifier(**LGB_PARAMS)

        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="binary_logloss",
            callbacks=[
                lgb.early_stopping(50, verbose=False),   # silent early stopping
                lgb.log_evaluation(period=0)             # no iteration logs
            ]
        )

        preds = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, preds)

        results.append({"auc": auc})

    return pd.DataFrame(results)

🔄 STEP 7: Running rolling CV...


In [27]:
# ==========================================================
# 🔹 STEP 8: MODEL COMPARISON
# ==========================================================

print("🚀 STEP 8: Running models...")

cv_hybrid = rolling_train(panel, hybrid_features)
cv_hybrid['model'] = "hybrid"

base_features = technical_features + price_structure_features
cv_base = rolling_train(panel, base_features)
cv_base['model'] = "tech_price"

cv_compare = pd.concat([cv_base, cv_hybrid], ignore_index=True)

if not cv_compare.empty:
    print("\n📈 Average AUC by Model:")
    print(cv_compare.groupby("model")['auc'].mean())
else:
    print("⚠️ No valid CV folds produced.")

🚀 STEP 8: Running models...


100%|██████████| 1100/1100 [08:49<00:00,  2.08it/s]


📈 Average AUC by Model:
model
hybrid        0.632668
tech_price    0.632600
Name: auc, dtype: float64


In [28]:
if not cv_compare.empty:

    print("\n📈 Average AUC by Model:")
    auc_summary = cv_compare.groupby("model")['auc'].mean()
    print(auc_summary)

    if 'hybrid' in auc_summary.index and 'tech_price' in auc_summary.index:

        auc_diff = auc_summary['hybrid'] - auc_summary['tech_price']

        print(f"\n🔍 AUC Difference (Hybrid - Base): {auc_diff:.6f}")

        print("\n🧠 Interpretation:")
        print("- AUC ~0.63 indicates a solid predictive edge in financial time-series.")
        print("- Difference close to 0.00 suggests current NLP features add minimal incremental signal.")
        print("- If Step 9 improves AUC meaningfully (>0.01), NLP timing alignment is working.")


📈 Average AUC by Model:
model
hybrid        0.632668
tech_price    0.632600
Name: auc, dtype: float64

🔍 AUC Difference (Hybrid - Base): 0.000069

🧠 Interpretation:
- AUC ~0.63 indicates a solid predictive edge in financial time-series.
- Difference close to 0.00 suggests current NLP features add minimal incremental signal.
- If Step 9 improves AUC meaningfully (>0.01), NLP timing alignment is working.


In [29]:
# ==========================================================
# 🔹 STEP 9: CONCALL SIGNAL ENHANCEMENT
# ==========================================================

print("🧠 STEP 9: Enhancing Concall Signals...")

panel = panel.sort_values(["ticker", "date"])

# ----------------------------------------------------------
# 1️⃣ Days Since Last Concall
# ----------------------------------------------------------

panel['days_since_call'] = (
    panel.groupby('ticker')['date']
    .transform(lambda x: (x - x.where(panel['guidance_delta'].notna())).ffill())
).dt.days

panel['days_since_call'] = panel['days_since_call'].fillna(999)

# ----------------------------------------------------------
# 2️⃣ Decay-Weighted Guidance Signal
# ----------------------------------------------------------

panel['guidance_decay'] = (
    panel['guidance_delta'] *
    np.exp(-panel['days_since_call'] / 30)
)

# ----------------------------------------------------------
# 3️⃣ Guidance Surprise (Change vs Previous Call)
# ----------------------------------------------------------

panel['prev_guidance'] = (
    panel.groupby('ticker')['guidance_delta']
    .shift(1)
)

panel['guidance_surprise'] = (
    panel['guidance_delta'] - panel['prev_guidance']
)

panel['guidance_surprise'] = panel['guidance_surprise'].fillna(0)

# ----------------------------------------------------------
# 4️⃣ Volatility Interaction
# ----------------------------------------------------------

if 'rolling_vol_20' in panel.columns:
    panel['guidance_vol_interaction'] = (
        panel['guidance_delta'] * panel['rolling_vol_20']
    )
else:
    panel['guidance_vol_interaction'] = 0

# ----------------------------------------------------------
# 5️⃣ Post-Concall 3-Day Reaction
# ----------------------------------------------------------

panel['post_call_return_3d'] = (
    panel.groupby('ticker')['close']
    .pct_change(3)
)

panel['post_call_return_3d'] = (
    panel['post_call_return_3d'] *
    (panel['days_since_call'] <= 3).astype(int)
)

panel['post_call_return_3d'] = panel['post_call_return_3d'].fillna(0)

# ----------------------------------------------------------
# Add new NLP-enhanced features to hybrid block
# ----------------------------------------------------------

enhanced_concall_features = [
    'guidance_decay',
    'guidance_surprise',
    'guidance_vol_interaction',
    'post_call_return_3d'
]

hybrid_features = hybrid_features + enhanced_concall_features

print(f"✅ Added {len(enhanced_concall_features)} enhanced concall features")
print(f"📊 Total Hybrid Features Now: {len(hybrid_features)}")

🧠 STEP 9: Enhancing Concall Signals...
✅ Added 4 enhanced concall features
📊 Total Hybrid Features Now: 18


In [30]:
# ==========================================================
# 🔹 STEP 10: RE-RUN MODEL WITH ENHANCED NLP FEATURES
# ==========================================================

print("🚀 STEP 10: Re-running models with enhanced NLP features...")

# ----------------------------------------------------------
# Define updated feature blocks
# ----------------------------------------------------------

base_features = technical_features + price_structure_features

enhanced_concall_features = [
    'guidance_decay',
    'guidance_surprise',
    'guidance_vol_interaction',
    'post_call_return_3d'
]

hybrid_features_enhanced = (
    technical_features +
    price_structure_features +
    concall_features +          # original guidance/confidence
    enhanced_concall_features   # new step 9 features
)

# Remove duplicates (safety)
hybrid_features_enhanced = list(set(hybrid_features_enhanced))

print(f"Base Features: {len(base_features)}")
print(f"Hybrid Features (Enhanced): {len(hybrid_features_enhanced)}")

# ----------------------------------------------------------
# Run Rolling CV
# ----------------------------------------------------------

cv_base = rolling_train(panel, base_features)
cv_base['model'] = "tech_price"

cv_hybrid = rolling_train(panel, hybrid_features_enhanced)
cv_hybrid['model'] = "hybrid_enhanced"

cv_compare = pd.concat([cv_base, cv_hybrid], ignore_index=True)

# ----------------------------------------------------------
# Print Results
# ----------------------------------------------------------

if not cv_compare.empty:

    auc_summary = cv_compare.groupby("model")['auc'].mean()

    print("\n📈 Average AUC by Model:")
    print(auc_summary)

    if "hybrid_enhanced" in auc_summary.index and "tech_price" in auc_summary.index:

        auc_diff = auc_summary["hybrid_enhanced"] - auc_summary["tech_price"]

        print(f"\n🔍 AUC Difference (Enhanced Hybrid - Base): {auc_diff:.6f}")

        print("\n🧠 Interpretation Guide:")
        print("• AUC ≈ 0.50 → Random")
        print("• AUC ≈ 0.60 → Weak but tradable")
        print("• AUC ≈ 0.63 → Solid retail quant edge")
        print("• AUC ≥ 0.65 → Strong signal")
        print("• ΔAUC < 0.005 → Negligible improvement")
        print("• ΔAUC 0.005–0.015 → Moderate lift")
        print("• ΔAUC > 0.02 → Significant NLP alpha")

else:
    print("⚠️ No valid CV folds produced.")

🚀 STEP 10: Re-running models with enhanced NLP features...
Base Features: 12
Hybrid Features (Enhanced): 17


100%|██████████| 1100/1100 [08:53<00:00,  2.06it/s]


📈 Average AUC by Model:
model
hybrid_enhanced    0.635083
tech_price         0.632600
Name: auc, dtype: float64

🔍 AUC Difference (Enhanced Hybrid - Base): 0.002483

🧠 Interpretation Guide:
• AUC ≈ 0.50 → Random
• AUC ≈ 0.60 → Weak but tradable
• AUC ≈ 0.63 → Solid retail quant edge
• AUC ≥ 0.65 → Strong signal
• ΔAUC < 0.005 → Negligible improvement
• ΔAUC 0.005–0.015 → Moderate lift
• ΔAUC > 0.02 → Significant NLP alpha


In [44]:
# ==========================================================
# 🔹 STEP 11 (FINAL STABLE VERSION)
#    STOCK-LEVEL AUC – TECH_PRICE vs NLP vs HYBRID
# ==========================================================

def rolling_train_stock_level_multi(panel,
                                     base_features,
                                     hybrid_features,
                                     min_train_days=120,
                                     test_window=20):

    # ------------------------------------------------------
    # Keep only features that actually exist in panel
    # ------------------------------------------------------
    base_features = [f for f in base_features if f in panel.columns]
    hybrid_features = [f for f in hybrid_features if f in panel.columns]

    # NLP features = hybrid minus base
    nlp_features = [f for f in hybrid_features if f not in base_features]

    if len(base_features) == 0 or len(hybrid_features) == 0:
        raise ValueError("No valid feature columns found in panel.")

    results = []
    dates = sorted(panel['date'].unique())

    for i in tqdm(range(min_train_days, len(dates) - test_window)):

        train = panel[panel['date'] <= dates[i]]
        test = panel[
            (panel['date'] > dates[i]) &
            (panel['date'] <= dates[i + test_window])
        ]

        if train.empty or test.empty:
            continue

        test = test.copy()

        model_feature_sets = {
            "tech_price": base_features,
            "nlp_only": nlp_features,
            "hybrid_enhanced": hybrid_features
        }

        for model_name, feature_set in model_feature_sets.items():

            if len(feature_set) == 0:
                continue  # skip if no valid features

            X_train = train[feature_set]
            y_train = train['target']

            X_test = test[feature_set]
            y_test = test['target']

            medians = X_train.median()
            X_train = X_train.fillna(medians)
            X_test = X_test.fillna(medians)

            model = LGBMClassifier(**LGB_PARAMS)

            model.fit(
                X_train, y_train,
                eval_set=[(X_test, y_test)],
                eval_metric="binary_logloss",
                callbacks=[
                    lgb.early_stopping(50, verbose=False),
                    lgb.log_evaluation(period=0)
                ]
            )

            test[f'pred_{model_name}'] = model.predict_proba(X_test)[:, 1]

        results.append(test)

    if not results:
        print("⚠️ No predictions generated.")
        return pd.DataFrame()

    all_preds = pd.concat(results, ignore_index=True)

    # ------------------------------------------------------
    # Compute AUC per stock
    # ------------------------------------------------------
    stock_auc = []

    for ticker, df in all_preds.groupby('ticker'):

        if df['target'].nunique() > 1:

            row = {"ticker": ticker}

            if 'pred_tech_price' in df:
                row["auc_tech_price"] = roc_auc_score(df['target'], df['pred_tech_price'])

            if 'pred_nlp_only' in df:
                row["auc_nlp_only"] = roc_auc_score(df['target'], df['pred_nlp_only'])

            if 'pred_hybrid_enhanced' in df:
                row["auc_hybrid_enhanced"] = roc_auc_score(df['target'], df['pred_hybrid_enhanced'])

            if "auc_tech_price" in row and "auc_hybrid_enhanced" in row:
                row["nlp_alpha"] = row["auc_hybrid_enhanced"] - row["auc_tech_price"]

            stock_auc.append(row)

    return pd.DataFrame(stock_auc)

In [45]:
stock_auc_df = rolling_train_stock_level_multi(
    panel,
    base_features,
    hybrid_features_enhanced
)

stock_auc_df.sort_values("auc_hybrid_enhanced", ascending=False).head(10)

100%|██████████| 1100/1100 [11:06<00:00,  1.65it/s]


,ticker,auc_tech_price,auc_hybrid_enhanced,nlp_alpha
19,BRITANNIA,0.915041,0.847623,-0.067418
68,PFC,0.793637,0.701510,-0.092128
66,NTPC,0.796031,0.684608,-0.111424
72,RECLTD,0.788712,0.663807,-0.124905
77,SHRIRAMFIN,0.796730,0.651866,-0.144864
48,IRFC,0.802878,0.643833,-0.159045
75,SBIN,0.670393,0.589786,-0.080607
78,SIEMENS,0.490257,0.589737,0.099480
76,SHREECEM,0.548712,0.577162,0.028450
61,MAXHEALTH,0.527042,0.575635,0.048593


In [48]:
stock_auc_df.sort_values("auc_hybrid_enhanced", ascending=False).tail(25)

,ticker,auc_tech_price,auc_hybrid_enhanced,nlp_alpha
64,NAUKRI,0.495557,0.500996,0.005438
5,ADANIPOWER,0.473137,0.499848,0.026712
90,TVSMOTOR,0.484743,0.498773,0.014030
6,AMBUJACEM,0.496679,0.497285,0.000606
15,BEL,0.499169,0.496549,-0.002620
69,PIDILITIND,0.539050,0.495631,-0.043420
88,TORNTPHARM,0.508020,0.495529,-0.012490
8,ASIANPAINT,0.513923,0.495268,-0.018654
58,LTIM,0.462326,0.494958,0.032632
14,BANKBARODA,0.485874,0.493917,0.008043
